## Import Library

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, GridSearchCV, train_test_split
from sklearn.linear_model import LassoCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import make_scorer
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import dask.dataframe as dd
import lightgbm as lgb
import optuna
# import xgboost as xgb
# import catboost as cb
import seaborn as sns
from scipy.stats import uniform, randint

## Model Building and Data Handling

### Define scoring using myscore

In [14]:
def weighted_mae_calculaion(y_true, y_pred, weights):
    errors = np.abs(y_true - y_pred) * weights
    weighted_errors = errors*weights
    return np.sum(weighted_errors) / np.sum(weights)
def weighted_mae(y_true, y_pred, weights):
    y_true = y_true.reset_index()
    y_true.columns = ['unique_id', 'sales']
    y_true = y_true.merge(weights, on = 'unique_id', how = 'left')
    weights = y_true['weight'].values
    y_true = y_true['sales'].values
    return weighted_mae_calculaion(y_true, y_pred, weights)

### Do PCA on data

In [15]:
def PCA_transformer(df_train, df_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(df_train)
    X_test_scaled = scaler.transform(df_test)
    pca_full = PCA().fit(X_train_scaled)
    cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
    optimal_components = np.argmax(cumulative_variance >= 0.95) + 1
    print(f"Optimal PCA Components: {optimal_components}")
    pca_optimal = PCA(n_components=optimal_components)
    X_train_pca = pca_optimal.fit_transform(X_train_scaled)
    X_test_pca = pca_optimal.transform(X_test_scaled)
    X_train_df = pd.DataFrame(X_train_pca, columns=[f'PC{i+1}' for i in range(optimal_components)])
    X_train_df['sales'] = df_train['sales'].values
    X_test_df = pd.DataFrame(X_test_pca, columns=[f'PC{i+1}' for i in range(optimal_components)])
    return X_train_df, X_test_df

### Get weight

In [16]:
# read from "test_weights.csv" using read.csv
weights = pd.read_csv("test_weights.csv")

## Model Training

### Import Data

In [17]:
# read from "processed_sales_test.csv" and processed_sales_train.csv using read.csv
sales_test = pd.read_csv("processed_sales_test2.csv")
sales_train = pd.read_csv("processed_sales_train2.csv")


In [18]:
# sales_test = sales_test.drop(columns = ['date'])
# sales_train = sales_train.drop(columns = ['date'])
# sales_test = sales_test.drop(columns = ['warehouse', 'holiday_name'])
# save sales_test to "processed_sales_test2.csv"
# sales_test.to_csv("processed_sales_test2.csv", index=False)
# sales_train.to_csv("processed_sales_train2.csv", index=False)

In [ ]:
pd.set_option("display.max_rows", None)  # Show all rows
pd.set_option("display.max_columns", None)  # Show all columns
print(sales_train.isnull().sum().reset_index())  # Check for missing values
# print((sales_train == np.inf).sum().reset_index())

                                                index  0
0                                           unique_id  0
1                                        total_orders  0
2                                               sales  0
3                                     sell_price_main  0
4                                   product_unique_id  0
5                                        shops_closed  0
6                              winter_school_holidays  0
7                                     school_holidays  0
8                                                year  0
9                                               month  0
10                                                day  0
11                                        day_of_week  0
12                                      new_years_day  0
13                           international_womens_day  0
14                                        good_friday  0
15                                      holy_saturday  0
16                             

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(sales_train.drop(columns = ['sales']), sales_train['sales'], test_size=0.2, random_state=42)
X_train = lgb.Dataset(X_train, y_train, params={"feature_pre_filter": False})
X_train.construct()
# X_test = lgb.Dataset(X_test, y_test, params={"feature_pre_filter": False})
# X_test.construct()
# X_train.save_binary("sales_train.bin")
# X_train = lgb.Dataset("sales_train.bin")
# X_test = lgb.Dataset(X_test, y_test)
# X_test.save_binary("sales_test.bin")
# X_test = lgb.Dataset("sales_test.bin")

### Train Model

In [66]:
def objective(trial):
    # for chunk in pd.read_csv("sales_train.csv", chunksize = 100000):
    #     X_chunk= chunk.drop(columns = ['sales'])
    #     y_chunk = chunk['sales']
    #   X_train, X_test, y_train, y_test = train_test_split(sales_train, sales_train, test_size=0.2, random_state=42)
    #     lgb_train = lgb.Dataset(X_train, y_train, free_raw_data=False)
    #     lgb_test = lgb.Dataset(X_test, y_test, free_raw_data=False)
    #     if lgb_model is None:
    #         lgb_model = lgb.train(param_distributions1, lgb_train, valid_sets=[lgb_test], num_boost_rounds = 100, early_stopping_rounds=10)
    #     else:
    #         lgb_model = lgb.train(param_distributions1, lgb_train, valid_sets=[lgb_test], num_boost_rounds = 100, init_model=lgb_model)
    #         lgb_model.reset_parameter({"num_boost_rounds": 100})
    try:
        param = {
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 300),
            "num_leaves": trial.suggest_int("num_leaves", 10, 200),
            "max_depth": trial.suggest_int("max_depth", 3, 20),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 50, 200),  # Corrected: Use suggest_int for integer values
            "subsample": trial.suggest_categorical("subsample", [0.7, 0.8, 0.9, 1.0]),
            "colsample_bytree": trial.suggest_categorical("colsample_bytree", [0.7, 0.8, 0.9, 1.0]),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 0.5, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 0.5, log=True)
            #"feature_pre_filter": False
        }
        model = lgb.train(param, X_train)
        y_pred = model.predict(X_test)
        wmae = weighted_mae(y_test, y_pred, weights)
        if pd.isna(y_pred).any():
                print(f"Trial {trial.number} failed due to NaN predictions.")
                return float("inf")
        return wmae
    except Exception as e:
         print(f"Trial {trial.number} failed due to error: {str(e)}")
         return float("inf")

In [68]:
# sales_train, sales_test = PCA_transformer(sales_train, sales_test)
# sales_train = lgb.Dataset(sales_train.drop(columns = ['sales']), sales_train['sales'])
# sales_train.save_binary("sales_train.bin")
# sales_train = lgb.Dataset("sales_train.bin")
pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
model = optuna.create_study(direction="minimize", pruner=pruner)
model.optimize(objective, n_trials = 100)
print("Best trial: score {},\nparams {}".format(model.best_trial.value, model.best_trial.params))
best_params = model.best_params
best_params["objective"] = "regression"
best_params["metric"] = "mae"
final_model = lgb.train(best_params, sales_train, num_boost_round=200)



[I 2025-02-13 01:31:46,773] A new study created in memory with name: no-name-5dbdede1-9ee4-44c5-92b9-a76c2e642b3b
c:\Users\NINGZHI_JIANG\anaconda3\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.089305 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1437
[LightGBM] [Info] Number of data points in the train set: 3205935, number of used features: 61
[LightGBM] [Info] Start training from score 108.251428


[W 2025-02-13 01:31:57,676] Trial 0 failed with parameters: {'min_data_in_leaf': 141, 'num_leaves': 193, 'max_depth': 14, 'learning_rate': 0.03357950242835994, 'n_estimators': 97, 'subsample': 0.7, 'colsample_bytree': 1.0, 'reg_alpha': 1.306953430374335e-07, 'reg_lambda': 0.003853982026543065} because of the following error: The value nan is not acceptable.
[W 2025-02-13 01:31:57,677] Trial 0 failed with value nan.
c:\Users\NINGZHI_JIANG\anaconda3\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.115288 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1437
[LightGBM] [Info] Number of data points in the train set: 3205935, number of used features: 61
[LightGBM] [Info] Start training from score 108.251428
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

[W 2025-02-13 01:32:02,931] Trial 1 failed with parameters: {'min_data_in_leaf': 20, 'num_leaves': 31, 'max_depth': 3, 'learning_rate': 0.019134624918618025, 'n_estimators': 136, 'subsample': 1.0, 'colsample_bytree': 0.9, 'reg_alpha': 2.694924041368697e-06, 'reg_lambda': 0.0855946528667375} because of the following error: The value nan is not acceptable.
[W 2025-02-13 01:32:02,932] Trial 1 failed with value nan.
c:\Users\NINGZHI_JIANG\anaconda3\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.123163 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1437
[LightGBM] [Info] Number of data points in the train set: 3205935, number of used features: 61
[LightGBM] [Info] Start training from score 108.251428


[W 2025-02-13 01:47:57,863] Trial 98 failed with parameters: {'min_data_in_leaf': 53, 'num_leaves': 163, 'max_depth': 14, 'learning_rate': 0.013059964485636738, 'n_estimators': 138, 'subsample': 0.8, 'colsample_bytree': 0.9, 'reg_alpha': 0.19034429122191784, 'reg_lambda': 0.0010766731421632508} because of the following error: The value nan is not acceptable.
[W 2025-02-13 01:47:57,864] Trial 98 failed with value nan.
c:\Users\NINGZHI_JIANG\anaconda3\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.142523 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1437
[LightGBM] [Info] Number of data points in the train set: 3205935, number of used features: 61
[LightGBM] [Info] Start training from score 108.251428
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

[W 2025-02-13 01:48:04,922] Trial 99 failed with parameters: {'min_data_in_leaf': 36, 'num_leaves': 110, 'max_depth': 5, 'learning_rate': 0.0997112661064452, 'n_estimators': 115, 'subsample': 1.0, 'colsample_bytree': 0.8, 'reg_alpha': 6.109822060000722e-07, 'reg_lambda': 2.5706302371197925e-05} because of the following error: The value nan is not acceptable.
[W 2025-02-13 01:48:04,923] Trial 99 failed with value nan.


ValueError: No trials are completed yet.

### Load the model and Make Prediction

In [ ]:
sales_pred = final_model.predict(sales_test)
pd.DataFrame({'sales': sales_pred}).to_csv("sales_pred.csv", index=False)
print("Predictions saved to predictions.csv")


In [ ]:
print(y_test)

884687     62.88
2246737    52.48
182212      5.88
436554     36.02
2166208    10.02
           ...  
1476853    35.00
360212      4.14
3079274    51.87
358347     85.37
3682922     3.86
Name: sales, Length: 801484, dtype: float64
